# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets
import sys
sys.path.append('../05_src/')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
import pypdf
from langchain_core.documents import Document

def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]

def get_document_content(path):
    docs = load_pdf_pages(path)
    document_text = ""
    for page in docs:
        document_text += page.page_content + "\n"
    return document_text

In [ ]:
file_path = "../05_src/documents/ai_report_2025.pdf"
content = get_document_content(file_path)

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from utils.clients import get_client
from pydantic import BaseModel
import os
from typing import List, Tuple, Optional
from openai import OpenAI

os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()

In [36]:
class DocumentSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [46]:
def get_article_summary(text: str, tone: str = "Legalese"):
    summarization_prompt = f'''
    You are an expert at structured document summarization.
    Produce a structured output using the requested schema.

    The summary must be written in the following distinguishable tone: {tone}.
    It must also be concise and no longer than 1000 tokens.
    '''

    completion = client.beta.chat.completions.parse(
        model=MODEL,
        temperature=0.2,
        messages=[
            {
                "role": "system",
                "content": summarization_prompt,
            },
            {
                "role": "user",
                "content": text,
            },
        ],
        response_format=DocumentSummary,
    )

    parsed = completion.choices[0].message.parsed

    parsed.InputTokens = completion.usage.prompt_tokens
    parsed.OutputTokens = completion.usage.completion_tokens

    return parsed


In [47]:
summary = get_article_summary(content)
summary

DocumentSummary(Author='MIT NANDA', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This report provides critical insights into the current state of Generative AI (GenAI) adoption and its impact on business transformation, highlighting the disparity between high adoption rates and low transformational outcomes across various sectors.', Summary="The report delineates the phenomenon termed the 'GenAI Divide,' wherein 95% of organizations experience negligible returns on substantial investments (estimated at $30-40 billion) in Generative AI technologies. Despite widespread adoption of tools like ChatGPT, only 5% of integrated AI pilots yield significant value, primarily due to inadequate learning capabilities and poor integration with existing workflows. The research identifies four key patterns contributing to this divide: limited disruption across sectors, an enterprise paradox of high pilot volume but low scalability, investment biases favoring visible functions over

In [48]:
def print_summary(summary):
    print(f"Author: {summary.Author}\n")
    print(f"Title: {summary.Title}\n")
    print(f"Relevance: {summary.Relevance}\n")
    print(f"Summary: {summary.Summary}\n")
    print(f"Tone: {summary.Tone}\n")
    print(f"Number of input tokens: {summary.InputTokens}")
    print(f"Number of output tokens: {summary.OutputTokens}")


In [49]:
print_summary(summary)


Author: MIT NANDA

Title: The GenAI Divide: State of AI in Business 2025

Relevance: This report provides critical insights into the current state of Generative AI (GenAI) adoption and its impact on business transformation, highlighting the disparity between high adoption rates and low transformational outcomes across various sectors.

Summary: The report delineates the phenomenon termed the 'GenAI Divide,' wherein 95% of organizations experience negligible returns on substantial investments (estimated at $30-40 billion) in Generative AI technologies. Despite widespread adoption of tools like ChatGPT, only 5% of integrated AI pilots yield significant value, primarily due to inadequate learning capabilities and poor integration with existing workflows. The research identifies four key patterns contributing to this divide: limited disruption across sectors, an enterprise paradox of high pilot volume but low scalability, investment biases favoring visible functions over high-ROI back-offi

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
# This is the original text to be summarized
input = content

# This is the summary, i.e. the actual output from the LLM application
actual_output = summary.Summary

In [50]:
summarization_questions = [
    "Does the summary correctly explain what the 'GenAI Divide' is and how it separates organizations with measurable AI value from those without ROI?",

    "Does the summary accurately report the key quantitative findings, including that only about 5% of organizations achieve meaningful production impact from enterprise GenAI systems?",

    "Does the summary distinguish between adoption of general-purpose tools like ChatGPT/Copilot and deployment of custom enterprise AI systems?",

    "Does the summary explain why most GenAI pilots fail to reach production, particularly the role of poor learning, memory, workflow integration, and contextual adaptation?",

    "Does the summary capture the report's argument that the main barrier to enterprise AI success is not model quality or regulation, but the inability of systems to learn and adapt over time?",

    "Does the summary describe the concept of the 'shadow AI economy' and explain how employees use personal AI tools more successfully than official enterprise systems?",

    "Does the summary accurately reflect the report's findings about investment allocation, including the concentration of GenAI budgets in sales and marketing despite stronger ROI in back-office automation?",

    "Does the summary describe the characteristics of organizations and vendors that successfully cross the GenAI Divide, including workflow customization, learning capability, and external partnerships?",

    "Does the summary preserve the report's discussion of workforce impact, including selective displacement in customer support and administrative functions rather than broad layoffs?",

    "Does the summary explain the report's vision of agentic AI and the 'Agentic Web,' including persistent memory, interoperable agents, and adaptive workflow coordination?"
]

coherence_questions = [
    "Does the summary present ideas in a logically organized order that mirrors the progression of the report?",

    "Does the summary clearly distinguish between general-purpose AI tools and enterprise AI systems without causing confusion?",

    "Does the summary explain the relationship between pilot failures, workflow integration problems, and the learning gap in a coherent manner?",

    "Does the summary maintain consistent terminology when discussing concepts such as the GenAI Divide, learning systems, and agentic AI?",

    "Does the summary avoid abrupt topic transitions and maintain readability across sections discussing adoption, ROI, workforce impact, and future trends?"
]

tonality_questions = [
    "Does the summary preserve the analytical and research-oriented tone of the original report?",

    "Does the summary avoid exaggerated hype or overly promotional language when discussing AI capabilities and future impact?",

    "Does the summary communicate skepticism and nuance around enterprise AI adoption challenges in a balanced way?",

    "Does the summary maintain a professional business and policy-oriented tone appropriate for executives and researchers?",

    "Does the summary appropriately balance optimism about agentic AI with the report's caution regarding current implementation failures?"
]

safety_questions = [
    "Does the summary avoid fabricating statistics, deployment rates, or ROI claims that are not supported by the report?",

    "Does the summary avoid presenting speculative future predictions as established facts?",

    "Does the summary avoid misrepresenting workforce impacts or overstating claims about job displacement caused by AI?",

    "Does the summary avoid attributing endorsements, opinions, or guarantees to organizations or technologies without evidence from the report?",

    "Does the summary preserve uncertainty, limitations, and methodological caveats discussed in the report rather than overstating conclusions?"
]

In [51]:
# from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

test_case = LLMTestCase(
  input = input,
  actual_output = actual_output
)

summarization_metric = SummarizationMetric(model = MODEL,
                                           assessment_questions=summarization_questions)


coherence_metric = GEval(
    name="Coherence",
    criteria=f"""
    Evaluate whether the summary is coherent, logically structured, and easy to follow.

    Assessment Questions: {coherence_questions}
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

tonality_metric = GEval(
    name="Tonality",
    criteria=f"""
    Evaluate whether the summary preserves the professional and analytical tone of the original report.

    Assessment Questions: {tonality_questions}
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

safety_metric = GEval(
    name="Safety",
    criteria=f"""
    Evaluate whether the summary is safe, factual, and avoids misleading claims.

    Assessment Questions: {safety_questions}
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)


/var/folders/36/lk67tbn1567_fc3pz503h6z00000gq/T/ipykernel_19731/3961299248.py:5: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


In [52]:
metrics = [
    summarization_metric,
    coherence_metric,
    tonality_metric,
    safety_metric,
]

for metric in metrics:
    metric.measure(test_case)

evaluation_output = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

evaluation_output

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.6363636363636364,
 'SummarizationReason': 'The score is 0.64 because the summary contains contradictions regarding investment amounts and introduces extra information not found in the original text, which affects its accuracy and completeness. Additionally, it leaves out key concepts and details that the original text addresses, leading to unanswered questions.',
 'CoherenceScore': 0.75,
 'CoherenceReason': 'The summary follows the report’s executive-summary progression well, moving from the core finding of high investment/low ROI into causes, patterns, and implications. It clearly distinguishes general-purpose tools like ChatGPT from enterprise-grade/custom systems and correctly notes that the former boost individual productivity while the latter often stall in pilots. It also captures key test-case details such as the $30–40B investment, 95% zero return, 5% of pilots generating value, the four divide patterns, and the importance of workflow integration and co

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [54]:
refinement_prompt = f"""
You are revising a document summary based on evaluation feedback.

Original document:
{content}

Previous summary:
{summary.Summary}

Evaluation feedback:
 - Summarization Score: {summarization_metric.score}
 - Reason: {summarization_metric.reason}

 - Coherence Score: {coherence_metric.score}
 - Reason: {coherence_metric.reason}

 - Tonality Score: {tonality_metric.score}
 - Reason: {tonality_metric.reason}

 - Safety Score: {safety_metric.score}
 - Reason: {safety_metric.reason}

Your goals are to:
- Address weaknesses identified in the evaluation
- Preserve factual accuracy
- Improve organization and readability
- Maintain a professional analytical tone
- Avoid hallucinations or unsupported claims
- Keep the summary concise

Generate an improved summary.
"""

In [55]:
improved_completion = client.chat.completions.create(
    model=MODEL,
    temperature=0.2,
    messages=[
        {
            "role": "developer",
            "content": "You are an expert report summarizer."
        },
        {
            "role": "user",
            "content": refinement_prompt
        }
    ]
)

improved_summary = improved_completion.choices[0].message.content

In [56]:
improved_test_case = LLMTestCase(
    input=content,
    actual_output=improved_summary
)

comparison = {
    'Original': {},
    'Improved': {}
}
for metric in metrics:
    comparison['Original'][metric] = metric.measure(test_case)
    comparison['Improved'][metric] = metric.measure(improved_test_case)

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()

In [57]:
comparison

{'Original': {<deepeval.metrics.summarization.summarization.SummarizationMetric at 0x11b48f3d0>: 0.5454545454545454,
  <deepeval.metrics.g_eval.g_eval.GEval at 0x11b53b6d0>: 0.7268941421369994,
  <deepeval.metrics.g_eval.g_eval.GEval at 0x11b522e10>: 0.7939913349825992,
  <deepeval.metrics.g_eval.g_eval.GEval at 0x11b4da5d0>: 0.8},
 'Improved': {<deepeval.metrics.summarization.summarization.SummarizationMetric at 0x11b48f3d0>: 0.5333333333333333,
  <deepeval.metrics.g_eval.g_eval.GEval at 0x11b53b6d0>: 0.7817574476193644,
  <deepeval.metrics.g_eval.g_eval.GEval at 0x11b522e10>: 0.7977022630089974,
  <deepeval.metrics.g_eval.g_eval.GEval at 0x11b4da5d0>: 0.8985936372956755}}

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
